# Практика 12. Довірчі інтервали для середнього і частки

**Варіант 1: Київ**

Мета: побудувати довірчий інтервал для середнього через t-розподіл, довірчий інтервал для частки через нормальне наближення та правильно інтерпретувати рівень довіри.

Для липня Києва беремо орієнтовний центр `21 °C` і генеруємо 30 денних температур.

In [1]:
import numpy as np
import pandas as pd
from scipy import stats

variant = 1
city = 'Київ'
july_mean = 21
np.random.seed(variant)
daily_temps = np.random.normal(loc=july_mean, scale=2.5, size=30)

print(f'Місто: {city}')
print(f'Липневий орієнтовний центр: {july_mean} °C')
print('daily_temps =', daily_temps)
print(f'Середнє: {daily_temps.mean():.4f} °C')
print(f'Вибіркове стандартне відхилення: {daily_temps.std(ddof=1):.4f} °C')

Місто: Київ
Липневий орієнтовний центр: 21 °C
daily_temps = [25.06086341 19.47060897 19.67957062 18.31757844 23.16351907 15.24615326
 25.36202941 19.09698275 21.79759774 20.37657406 24.65526984 15.84964823
 20.19395699 20.03986411 23.83442361 18.25027183 20.56892948 18.80535396
 21.10553437 22.45703803 18.24845206 23.86180927 23.2539768  22.25623585
 23.25213987 19.29068035 20.69277444 18.66057641 20.3302798  22.32588867]
Середнє: 20.8502 °C
Вибіркове стандартне відхилення: 2.5659 °C


## Завдання 1. Вибірка денних температур

Використано 30 значень, згенерованих навколо липневої температури Києва з фіксованим seed для відтворюваності. `ddof=1` дає вибіркове стандартне відхилення, яке оцінює невідоме генеральне σ.

## Завдання 2. Довірчий інтервал для середньої температури

Оскільки справжнє стандартне відхилення генеральної сукупності невідоме, використовуємо t-розподіл із `n-1` ступенями свободи.

In [2]:
mean = daily_temps.mean()
s = daily_temps.std(ddof=1)
n = len(daily_temps)
se = s / np.sqrt(n)
ci_mean_95 = stats.t.interval(0.95, df=n - 1, loc=mean, scale=se)

print(f'n = {n}')
print(f'x̄ = {mean:.4f} °C')
print(f's = {s:.4f} °C')
print(f'SE = {se:.4f} °C')
print(f'95% ДІ для середньої: ({ci_mean_95[0]:.4f}, {ci_mean_95[1]:.4f}) °C')

n = 30
x̄ = 20.8502 °C
s = 2.5659 °C
SE = 0.4685 °C
95% ДІ для середньої: (19.8920, 21.8083) °C


**Пояснення.** z-критичне значення було б доречним, якби генеральне σ було відоме або вибірка була достатньо великою для надійного нормального наближення. Тут `n=30`, а σ невідоме й замінене оцінкою `s`, тому додаткова невизначеність враховується t-розподілом із 29 ступенями свободи.

## Завдання 3. Довірчий інтервал для частки

Розглянемо частку днів, коли температура перевищує табличне липневе значення 21 °C.

In [3]:
above_threshold = daily_temps > july_mean
p_hat = above_threshold.mean()
n_success = n * p_hat
n_failure = n * (1 - p_hat)

se_p = np.sqrt(p_hat * (1 - p_hat) / n)
ci_prop_95 = stats.norm.interval(0.95, loc=p_hat, scale=se_p)

print(f'Кількість днів вище {july_mean} °C: {above_threshold.sum()} із {n}')
print(f'p_hat = {p_hat:.4f}')
print(f'n * p_hat = {n_success:.2f}')
print(f'n * (1 - p_hat) = {n_failure:.2f}')
print(f'95% ДІ для частки: ({ci_prop_95[0]:.4f}, {ci_prop_95[1]:.4f})')
print(f'Чи потрапляє теоретичне значення 0.5 в інтервал? {ci_prop_95[0] <= 0.5 <= ci_prop_95[1]}')

Кількість днів вище 21 °C: 13 із 30
p_hat = 0.4333
n * p_hat = 13.00
n * (1 - p_hat) = 17.00
95% ДІ для частки: (0.2560, 0.6107)
Чи потрапляє теоретичне значення 0.5 в інтервал? True


**Висновок.** Для нормального наближення перевіряємо, що `n*p_hat` і `n*(1-p_hat)` достатньо великі, зазвичай не менші за 5. У цій вибірці обидві кількості задовольняють умову, тому наближення прийнятне. Теоретична частка для симетричної генерації дорівнює приблизно 0.5. Якщо 0.5 не потрапив до конкретного інтервалу, це можливо через випадковість однієї вибірки: при повторенні процедури близько 95% побудованих інтервалів накриватимуть справжню частку.

## Завдання 4. Ширина інтервалу при різних рівнях довіри

In [4]:
confidence_levels = [0.90, 0.95, 0.99]
interval_rows = []
for confidence in confidence_levels:
    interval = stats.t.interval(confidence, df=n - 1, loc=mean, scale=se)
    interval_rows.append({
        'рівень довіри': confidence,
        'нижня межа': interval[0],
        'верхня межа': interval[1],
        'ширина': interval[1] - interval[0]
    })

interval_table = pd.DataFrame(interval_rows)
display(interval_table.style.format({'рівень довіри': '{:.0%}', 'нижня межа': '{:.4f}', 'верхня межа': '{:.4f}', 'ширина': '{:.4f}'}))

,рівень довіри,нижня межа,верхня межа,ширина
0,90%,20.0542,21.6461,1.5920
1,95%,19.8920,21.8083,1.9163
2,99%,19.5589,22.1414,2.5826


**Висновок.** Ширина інтервалу зростає в порядку 90%, 95%, 99%, бо вищий рівень довіри потребує більшого критичного коефіцієнта. При незмінному рівні довіри вужчий інтервал можна отримати, зменшивши стандартну похибку, тобто збільшивши розмір вибірки або зменшивши варіативність даних.

## Завдання 5. Інтерпретація результату словами

Обчислений у Завданні 2 95% довірчий інтервал для середньої температури є діапазоном, отриманим за цією конкретною вибіркою. Це не означає, що вже обчислений інтервал має 95% ймовірності містити фіксоване істинне середнє. Якби багато разів генерувати нові вибірки по 30 днів і щоразу будувати інтервал тим самим методом, приблизно 95% таких інтервалів накривали б істинне середнє генеральної сукупності.

## Контрольні питання

**1. Точкова оцінка і довірчий інтервал.** Точкова оцінка, наприклад `x̄`, дає одне найкраще наближене число, але не показує його невизначеність. Довірчий інтервал додає діапазон можливих значень і враховує розмір вибірки та варіативність даних.

**2. Чому t-розподіл?** Коли σ невідоме, його замінюють вибірковим `s`. За невеликої або помірної вибірки ця заміна додає невизначеність, яку враховує t-розподіл. При зростанні числа ступенів свободи він наближається до стандартного нормального розподілу.

**3. Що означає 95% довіри?** Це властивість процедури: при багаторазовому повторенні вибірки та побудові інтервалів одним способом приблизно 95% інтервалів накриють істинний параметр. Помилково говорити про 95% ймовірності для істинного значення після того, як конкретний інтервал уже обчислено.

**4. Від чого залежить ширина?** На ширину впливають рівень довіри, стандартна похибка та критичне значення розподілу. Вищий рівень довіри збільшує інтервал, більша варіативність збільшує його, а більший розмір вибірки зменшує стандартну похибку і робить інтервал вужчим.

## Підсумок

Для вибірки денних температур Києва побудовано t-інтервал для середнього та інтервал для частки через нормальне наближення. Перевірено умови застосовності, порівняно ширину інтервалів при 90%, 95% і 99% довірі та сформульовано правильну частотну інтерпретацію довірчого інтервалу.